# Módulo 4 — Diseño e Implementación de Modelos de Análisis de Supervivencia
**Dataset:** METABRIC — Breast Cancer  
**Modelos:** Kaplan-Meier (referencia) · Cox PH · Random Survival Forest · DeepSurv  
**Métricas:** C-index · Integrated Brier Score · Brier Score temporal  
**Validación:** 5-fold Cross-Validation estratificada

---
## 0. Imports y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from itertools import combinations

# scikit-survival
from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import (
    concordance_index_censored,
    brier_score,
    integrated_brier_score
)

# lifelines (forest plot Cox)
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
from matplotlib.lines import Line2D

# Validación cruzada
from sklearn.model_selection import StratifiedKFold

# Deep Learning
import torch
import torch.nn as nn
import torchtuples as tt
from pycox.models import CoxPH as DeepSurvModel

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

os.makedirs('../images/Modelos', exist_ok=True)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {"disponible" if torch.cuda.is_available() else "no disponible — CPU"}')

In [ ]:
# ── Carga de datos preprocesados ──────────────────────────────────────────────
SAVE_DIR = '../data/processed/metabric'

X_train_np = np.load(f'{SAVE_DIR}/X_train_np.npy')
X_test_np  = np.load(f'{SAVE_DIR}/X_test_np.npy')
dur_train  = np.load(f'{SAVE_DIR}/dur_train.npy')
dur_test   = np.load(f'{SAVE_DIR}/dur_test.npy')
evt_train  = np.load(f'{SAVE_DIR}/evt_train.npy')
evt_test   = np.load(f'{SAVE_DIR}/evt_test.npy')
y_train    = joblib.load(f'{SAVE_DIR}/y_train.pkl')
y_test     = joblib.load(f'{SAVE_DIR}/y_test.pkl')
X_train    = pd.read_parquet(f'{SAVE_DIR}/X_train_df.parquet')
X_test     = pd.read_parquet(f'{SAVE_DIR}/X_test_df.parquet')

# Rango temporal de evaluación (percentil 10–90, evita extrapolación)
times_eval = np.percentile(dur_test, np.linspace(10, 90, 100))
times_eval = times_eval[(times_eval > dur_test.min()) & (times_eval < dur_test.max())]

# IBS de referencia KM (calculado en preprocesado)
IBS_KM_REF = 0.2156

print('✓ Datos cargados correctamente.')
print(f'  X_train_np : {X_train_np.shape}')
print(f'  X_test_np  : {X_test_np.shape}')
print(f'  y_train    : {y_train.dtype}  {y_train.shape}')
print(f'  Tasa eventos train : {evt_train.mean():.2%}')
print(f'  Tasa eventos test  : {evt_test.mean():.2%}')
print(f'  IBS KM referencia  : {IBS_KM_REF}')

---
## 1. Configuración del esquema de validación cruzada

Se utiliza **5-fold Cross-Validation estratificada** por el indicador de evento para garantizar que cada fold mantenga la misma tasa de censura que la cohorte global. Todos los modelos utilizan el mismo esquema de folds para que las comparaciones sean válidas.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Diccionario para acumular resultados de todos los modelos
resultados_cv  = {}   # C-index por fold
resultados_ibs = {}   # IBS por fold
resultados_test = {}  # métricas en test

print(f'Esquema de CV: {N_FOLDS}-fold estratificado')
print(f'Folds generados sobre {len(X_train_np)} registros de train')

---
## 2. Modelo de Riesgos Proporcionales de Cox (Cox PH)

El modelo de Cox es el modelo semiparamétrico estándar en análisis de supervivencia clínico. Modela el *hazard* instantáneo como:

$$h(t|x) = h_0(t) \cdot \exp(\beta^\top x)$$

donde $h_0(t)$ es el *hazard* basal no especificado y $\beta$ son los coeficientes aprendidos. El supuesto central es que los *hazard ratios* entre grupos son **constantes en el tiempo** (proporcionalidad de riesgos).

Se usa regularización L2 (`alpha=0.1`) para estabilizar los coeficientes dado el número de covariables (70) relativo al tamaño muestral.

### 2.1. Validación cruzada 5-fold

In [ ]:
cox_cv_cindex = []
cox_cv_ibs    = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_np, evt_train.astype(int))):

    X_tr,  X_val  = X_train_np[tr_idx], X_train_np[val_idx]
    y_tr   = Surv.from_arrays(evt_train[tr_idx].astype(bool),  dur_train[tr_idx])
    y_val  = Surv.from_arrays(evt_train[val_idx].astype(bool), dur_train[val_idx])

    cox = CoxPHSurvivalAnalysis(alpha=0.1, ties='efron')
    cox.fit(X_tr, y_tr)

    # C-index
    ci = cox.score(X_val, y_val)
    cox_cv_cindex.append(ci)

    # IBS
    t_val = np.percentile(dur_train[val_idx], np.linspace(10, 90, 50))
    t_val = t_val[(t_val > dur_train[val_idx].min()) & (t_val < dur_train[val_idx].max())]
    surv_fns = cox.predict_survival_function(X_val)
    preds    = np.row_stack([fn(t_val) for fn in surv_fns])
    ibs      = integrated_brier_score(y_tr, y_val, preds, t_val)
    cox_cv_ibs.append(ibs)

    print(f'  Fold {fold+1}/{N_FOLDS}  C-index: {ci:.4f}  IBS: {ibs:.4f}')

resultados_cv['Cox PH']  = cox_cv_cindex
resultados_ibs['Cox PH'] = cox_cv_ibs

print(f'\n── Resultados CV — Cox PH ─────────────────────────')
print(f'  C-index : {np.mean(cox_cv_cindex):.4f} ± {np.std(cox_cv_cindex):.4f}')
print(f'  IBS     : {np.mean(cox_cv_ibs):.4f} ± {np.std(cox_cv_ibs):.4f}')

### 2.2. Modelo final y evaluación en test

In [ ]:
cox_final = CoxPHSurvivalAnalysis(alpha=0.1, ties='efron')
cox_final.fit(X_train_np, y_train)

cox_test_cindex = cox_final.score(X_test_np, y_test)

surv_fns_cox = cox_final.predict_survival_function(X_test_np)
preds_cox    = np.row_stack([fn(times_eval) for fn in surv_fns_cox])
cox_test_ibs = integrated_brier_score(y_train, y_test, preds_cox, times_eval)

resultados_test['Cox PH'] = {'C-index': cox_test_cindex, 'IBS': cox_test_ibs}

print(f'── Evaluación en TEST — Cox PH ─────────────────────')
print(f'  C-index : {cox_test_cindex:.4f}')
print(f'  IBS     : {cox_test_ibs:.4f}  (ref. KM: {IBS_KM_REF:.4f})')

### 2.3. Forest plot — Hazard Ratios

Los coeficientes del modelo de Cox se expresan como **Hazard Ratios** (HR): un HR > 1 indica que la variable aumenta el riesgo de muerte; HR < 1 indica efecto protector. Se muestran las 20 variables más significativas.

In [ ]:
df_cox_lf = pd.DataFrame(X_train_np, columns=X_train.columns)
df_cox_lf['duration'] = dur_train
df_cox_lf['event']    = evt_train.astype(int)

cph_lf = CoxPHFitter(penalizer=0.1)
cph_lf.fit(df_cox_lf, duration_col='duration', event_col='event', show_progress=False)

summary = cph_lf.summary.copy()
summary['HR']    = np.exp(summary['coef'])
summary['HR_lo'] = np.exp(summary['coef lower 95%'])
summary['HR_hi'] = np.exp(summary['coef upper 95%'])
top20 = summary.nsmallest(20, 'p').sort_values('HR')

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['firebrick' if hr > 1 else 'steelblue' for hr in top20['HR']]
ax.barh(top20.index, top20['HR'] - 1, left=1, color=colors, alpha=0.7, height=0.6)
ax.errorbar(
    top20['HR'], range(len(top20)),
    xerr=[top20['HR'] - top20['HR_lo'], top20['HR_hi'] - top20['HR']],
    fmt='none', color='black', capsize=3, linewidth=1.2
)
ax.axvline(1, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
ax.set_xlabel('Hazard Ratio (IC 95%)', fontsize=11)
ax.set_title('Forest Plot — Cox PH\nTop 20 variables por significancia (p-valor)',
             fontsize=12, fontweight='bold')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f"{v}  (p={summary.loc[v,'p']:.3f})" for v in top20.index], fontsize=8)
ax.grid(axis='x', alpha=0.3)
legend_elements = [
    Line2D([0],[0], color='firebrick', linewidth=6, alpha=0.7, label='HR > 1 (mayor riesgo)'),
    Line2D([0],[0], color='steelblue', linewidth=6, alpha=0.7, label='HR < 1 (factor protector)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../images/Modelos/Cox_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4. Verificación del supuesto de proporcionalidad de riesgos

El test de Schoenfeld evalúa si el *hazard ratio* de cada variable es constante en el tiempo. Un **p < 0.05** indica que la variable viola el supuesto de proporcionalidad, lo que sesga los coeficientes de Cox para esa variable.

In [ ]:
ph_test    = proportional_hazard_test(cph_lf, df_cox_lf, time_transform='rank')
ph_summary = ph_test.summary.copy()

violaciones = ph_summary[ph_summary['p'] < 0.05].sort_values('p')
cumplen     = ph_summary[ph_summary['p'] >= 0.05]

print(f'── Test de Schoenfeld ───────────────────────────────')
print(f'  Variables que CUMPLEN PH  : {len(cumplen)}')
print(f'  Variables que VIOLAN PH   : {len(violaciones)}')
print(f'\nTop 10 violaciones (p < 0.05):')
display(violaciones[['test_statistic', 'p']].head(10).round(4))

---
## 3. Random Survival Forest (RSF)

RSF es una extensión del algoritmo Random Forest al contexto de la supervivencia. Construye un *ensemble* de árboles de supervivencia utilizando el estadístico **log-rank** como criterio de división de nodos. Sus ventajas frente a Cox son:
- No asume proporcionalidad de riesgos
- Captura relaciones no lineales e interacciones entre variables
- Robusto frente a outliers residuales

Referencia: Ishwaran et al., *The Annals of Applied Statistics*, 2008.

### 3.1. Validación cruzada 5-fold

In [ ]:
RSF_PARAMS = dict(
    n_estimators     = 200,
    min_samples_leaf = 15,
    max_features     = 'sqrt',
    n_jobs           = -1,
    random_state     = RANDOM_STATE
)

rsf_cv_cindex = []
rsf_cv_ibs    = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_np, evt_train.astype(int))):

    X_tr,  X_val  = X_train_np[tr_idx], X_train_np[val_idx]
    y_tr   = Surv.from_arrays(evt_train[tr_idx].astype(bool),  dur_train[tr_idx])
    y_val  = Surv.from_arrays(evt_train[val_idx].astype(bool), dur_train[val_idx])

    rsf = RandomSurvivalForest(**RSF_PARAMS)
    rsf.fit(X_tr, y_tr)

    ci = rsf.score(X_val, y_val)
    rsf_cv_cindex.append(ci)

    t_val = np.percentile(dur_train[val_idx], np.linspace(10, 90, 50))
    t_val = t_val[(t_val > dur_train[val_idx].min()) & (t_val < dur_train[val_idx].max())]
    surv_fns = rsf.predict_survival_function(X_val)
    preds    = np.row_stack([fn(t_val) for fn in surv_fns])
    ibs      = integrated_brier_score(y_tr, y_val, preds, t_val)
    rsf_cv_ibs.append(ibs)

    print(f'  Fold {fold+1}/{N_FOLDS}  C-index: {ci:.4f}  IBS: {ibs:.4f}')

resultados_cv['RSF']  = rsf_cv_cindex
resultados_ibs['RSF'] = rsf_cv_ibs

print(f'\n── Resultados CV — RSF ─────────────────────────────')
print(f'  C-index : {np.mean(rsf_cv_cindex):.4f} ± {np.std(rsf_cv_cindex):.4f}')
print(f'  IBS     : {np.mean(rsf_cv_ibs):.4f} ± {np.std(rsf_cv_ibs):.4f}')

### 3.2. Modelo final y evaluación en test

In [ ]:
rsf_final = RandomSurvivalForest(**RSF_PARAMS)
rsf_final.fit(X_train_np, y_train)

rsf_test_cindex = rsf_final.score(X_test_np, y_test)

surv_fns_rsf = rsf_final.predict_survival_function(X_test_np)
preds_rsf    = np.row_stack([fn(times_eval) for fn in surv_fns_rsf])
rsf_test_ibs = integrated_brier_score(y_train, y_test, preds_rsf, times_eval)

resultados_test['RSF'] = {'C-index': rsf_test_cindex, 'IBS': rsf_test_ibs}

print(f'── Evaluación en TEST — RSF ────────────────────────')
print(f'  C-index : {rsf_test_cindex:.4f}')
print(f'  IBS     : {rsf_test_ibs:.4f}  (ref. KM: {IBS_KM_REF:.4f})')

### 3.3. Feature Importance

La importancia se basa en la reducción del error de predicción al permutar aleatoriamente cada variable. Una variable importante produce una caída grande del C-index cuando se permuta.

In [ ]:
importances = rsf_final.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors  = ['steelblue' if i < 5 else 'lightsteelblue' for i in range(len(feat_imp_df))]
ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1],
        color=colors[::-1], alpha=0.85)
ax.set_xlabel('Importancia (reducción del error por permutación)', fontsize=11)
ax.set_title('Random Survival Forest — Top 20 Variables\nMETABRIC',
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../images/Modelos/RSF_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. DeepSurv

DeepSurv (Katzman et al., 2018) extiende el modelo de Cox reemplazando la función lineal $\beta^\top x$ por una **red neuronal feedforward** que aprende relaciones no lineales entre las covariables y el log-riesgo. La función de pérdida es la log-verosimilitud parcial negativa de Cox.

**Arquitectura:**
```
Input(70) → Linear(256) → BN → ReLU → Dropout(0.3)
          → Linear(128) → BN → ReLU → Dropout(0.3)
          → Linear(64)  → BN → ReLU
          → Linear(1)   [log-riesgo]
```

Implementación mediante `pycox` (Kvamme et al.).

### 4.1. Arquitectura y funciones auxiliares

In [ ]:
def build_deepsurv_net(in_features: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 128),         nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64),          nn.BatchNorm1d(64),  nn.ReLU(),
        nn.Linear(64, 1)
    )

def to_pycox(X: np.ndarray, durations: np.ndarray, events: np.ndarray):
    return X.astype(np.float32), (durations.astype(np.float32), events.astype(np.float32))

DEEPSURV_PARAMS = dict(lr=1e-3, batch_size=256, epochs=100, patience=15)

print('✓ Arquitectura DeepSurv definida.')
print(build_deepsurv_net(X_train_np.shape[1]))

### 4.2. Validación cruzada 5-fold

In [ ]:
ds_cv_cindex = []
ds_cv_ibs    = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_np, evt_train.astype(int))):

    x_tr,  y_tr_ds  = to_pycox(X_train_np[tr_idx],  dur_train[tr_idx],  evt_train[tr_idx])
    x_val, y_val_ds = to_pycox(X_train_np[val_idx], dur_train[val_idx], evt_train[val_idx])

    net   = build_deepsurv_net(x_tr.shape[1])
    model = DeepSurvModel(net, tt.optim.Adam(lr=DEEPSURV_PARAMS['lr']))

    log = model.fit(
        x_tr, y_tr_ds,
        batch_size = DEEPSURV_PARAMS['batch_size'],
        epochs     = DEEPSURV_PARAMS['epochs'],
        callbacks  = [tt.callbacks.EarlyStopping(patience=DEEPSURV_PARAMS['patience'])],
        val_data   = (x_val, y_val_ds),
        verbose    = False
    )

    ci = model.concordance_td(x_val, y_val_ds)
    ds_cv_cindex.append(ci)

    # IBS
    y_tr_surv  = Surv.from_arrays(evt_train[tr_idx].astype(bool),  dur_train[tr_idx])
    y_val_surv = Surv.from_arrays(evt_train[val_idx].astype(bool), dur_train[val_idx])
    t_val = np.percentile(dur_train[val_idx], np.linspace(10, 90, 50))
    t_val = t_val[(t_val > dur_train[val_idx].min()) & (t_val < dur_train[val_idx].max())]
    surv_df = model.predict_surv_df(x_val)
    preds   = np.row_stack([
        np.interp(t_val, surv_df.index.values, surv_df.iloc[:, i].values)
        for i in range(surv_df.shape[1])
    ])
    ibs = integrated_brier_score(y_tr_surv, y_val_surv, preds, t_val)
    ds_cv_ibs.append(ibs)

    print(f'  Fold {fold+1}/{N_FOLDS}  C-index: {ci:.4f}  IBS: {ibs:.4f}  '
          f'Epochs: {len(log.to_pandas())}')

resultados_cv['DeepSurv']  = ds_cv_cindex
resultados_ibs['DeepSurv'] = ds_cv_ibs

print(f'\n── Resultados CV — DeepSurv ────────────────────────')
print(f'  C-index : {np.mean(ds_cv_cindex):.4f} ± {np.std(ds_cv_cindex):.4f}')
print(f'  IBS     : {np.mean(ds_cv_ibs):.4f} ± {np.std(ds_cv_ibs):.4f}')

### 4.3. Modelo final, curva de aprendizaje y evaluación en test

In [ ]:
x_train_ds, y_train_ds = to_pycox(X_train_np, dur_train, evt_train)
x_test_ds,  y_test_ds  = to_pycox(X_test_np,  dur_test,  evt_test)

net_final   = build_deepsurv_net(X_train_np.shape[1])
model_final = DeepSurvModel(net_final, tt.optim.Adam(lr=DEEPSURV_PARAMS['lr']))

log_final = model_final.fit(
    x_train_ds, y_train_ds,
    batch_size = DEEPSURV_PARAMS['batch_size'],
    epochs     = DEEPSURV_PARAMS['epochs'],
    callbacks  = [tt.callbacks.EarlyStopping(patience=DEEPSURV_PARAMS['patience'])],
    val_data   = (x_test_ds, y_test_ds),
    verbose    = True
)

# Curva de aprendizaje
log_df = log_final.to_pandas()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(log_df['train_loss'], label='Train loss', color='steelblue', linewidth=1.8)
ax.plot(log_df['val_loss'],   label='Val loss',   color='firebrick', linewidth=1.8, linestyle='--')
ax.set_xlabel('Época', fontsize=11)
ax.set_ylabel('Pérdida (neg. log-likelihood parcial de Cox)', fontsize=11)
ax.set_title('DeepSurv — Curva de aprendizaje', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../images/Modelos/DeepSurv_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Métricas en test
ds_test_cindex = model_final.concordance_td(x_test_ds, y_test_ds)

surv_df_test = model_final.predict_surv_df(x_test_ds)
preds_ds     = np.row_stack([
    np.interp(times_eval, surv_df_test.index.values, surv_df_test.iloc[:, i].values)
    for i in range(surv_df_test.shape[1])
])
ds_test_ibs = integrated_brier_score(y_train, y_test, preds_ds, times_eval)

resultados_test['DeepSurv'] = {'C-index': ds_test_cindex, 'IBS': ds_test_ibs}

print(f'\n── Evaluación en TEST — DeepSurv ───────────────────')
print(f'  C-index       : {ds_test_cindex:.4f}')
print(f'  IBS           : {ds_test_ibs:.4f}  (ref. KM: {IBS_KM_REF:.4f})')
print(f'  Épocas entrend: {len(log_df)}')

---
## 5. Comparativa de rendimiento entre modelos

**Guía de interpretación de métricas:**
- **C-index:** probabilidad de ordenación correcta del riesgo. 0.5 = azar · >0.70 = aceptable · >0.75 = bueno
- **IBS:** error de calibración integrado. 0.25 = azar · <0.15 = bueno · <0.10 = excelente
- **Referencia KM:** IBS = 0.2156 — cualquier modelo debe superarlo para justificar el uso de covariables

### 5.1. Tabla comparativa

In [ ]:
rows = []
for modelo in ['Cox PH', 'RSF', 'DeepSurv']:
    cv_ci  = resultados_cv[modelo]
    cv_ibs = resultados_ibs[modelo]
    rows.append({
        'Modelo'        : modelo,
        'C-index CV'    : f"{np.mean(cv_ci):.4f} ± {np.std(cv_ci):.4f}",
        'IBS CV'        : f"{np.mean(cv_ibs):.4f} ± {np.std(cv_ibs):.4f}",
        'C-index Test'  : f"{resultados_test[modelo]['C-index']:.4f}",
        'IBS Test'      : f"{resultados_test[modelo]['IBS']:.4f}",
        'Supera KM'     : '✓' if resultados_test[modelo]['IBS'] < IBS_KM_REF else '✗'
    })

# Añadir KM como referencia
rows.insert(0, {
    'Modelo'       : 'KM (referencia)',
    'C-index CV'   : '—',
    'IBS CV'       : '—',
    'C-index Test' : '—',
    'IBS Test'     : f'{IBS_KM_REF:.4f}',
    'Supera KM'    : '—'
})

df_comparativa = pd.DataFrame(rows).set_index('Modelo')
display(df_comparativa)

### 5.2. Boxplot de C-index e IBS por fold (CV)

In [ ]:
modelos  = ['Cox PH', 'RSF', 'DeepSurv']
colores  = ['#4C72B0', '#55A868', '#C44E52']
cv_ci    = [resultados_cv[m]  for m in modelos]
cv_ibs_l = [resultados_ibs[m] for m in modelos]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, ylabel, ref, ref_label, invert in [
    (axes[0], cv_ci,    'C-index',                 0.5,       'Azar (0.5)',  False),
    (axes[1], cv_ibs_l, 'Integrated Brier Score',  IBS_KM_REF,'Ref. KM',    True),
]:
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(ref, color='gray', linestyle='--', linewidth=1.2, label=ref_label)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(modelos, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f'{ylabel} por fold (5-fold CV)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    if invert:
        ax.invert_yaxis()

plt.suptitle('Comparativa de modelos — Validación cruzada 5-fold\nMETABRIC',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/Modelos/Comparativa_CV_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3. Brier Score temporal

El Brier Score puntual $B(t)$ revela en qué horizonte temporal cada modelo es más o menos preciso, lo que el IBS escalar no puede mostrar.

In [ ]:
_, bs_cox = brier_score(y_train, y_test, preds_cox, times_eval)
_, bs_rsf = brier_score(y_train, y_test, preds_rsf, times_eval)
_, bs_ds  = brier_score(y_train, y_test, preds_ds,  times_eval)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(times_eval, bs_cox, label=f'Cox PH   (IBS={resultados_test["Cox PH"]["IBS"]:.3f})',
        color='#4C72B0', linewidth=2)
ax.plot(times_eval, bs_rsf, label=f'RSF      (IBS={resultados_test["RSF"]["IBS"]:.3f})',
        color='#55A868', linewidth=2)
ax.plot(times_eval, bs_ds,  label=f'DeepSurv (IBS={resultados_test["DeepSurv"]["IBS"]:.3f})',
        color='#C44E52', linewidth=2, linestyle='--')
ax.axhline(IBS_KM_REF, color='gray', linestyle=':', linewidth=1.5,
           label=f'KM marginal — referencia nula (IBS={IBS_KM_REF:.3f})')
ax.axhline(0.25, color='lightgray', linestyle=':', linewidth=1,
           label='Azar puro (0.25)')
ax.set_xlabel('Tiempo (meses)', fontsize=12)
ax.set_ylabel('Brier Score  B(t)', fontsize=12)
ax.set_title('Brier Score temporal — Conjunto Test\nMETABRIC',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 0.30)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../images/Modelos/Comparativa_Brier_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4. Resumen ejecutivo final

In [ ]:
mejor_ci  = max(resultados_test.items(), key=lambda x: x[1]['C-index'])
mejor_ibs = min(resultados_test.items(), key=lambda x: x[1]['IBS'])

print('═' * 65)
print('  COMPARATIVA FINAL — METABRIC')
print('═' * 65)
print(f'  {"Modelo":<12} {"C-index CV":>16} {"IBS CV":>14} {"C-index Test":>14} {"IBS Test":>10}')
print('  ' + '─' * 60)
for m in modelos:
    ci_cv  = resultados_cv[m]
    ibs_cv = resultados_ibs[m]
    print(f'  {m:<12} '
          f'{np.mean(ci_cv):>8.4f}±{np.std(ci_cv):.4f} '
          f'{np.mean(ibs_cv):>7.4f}±{np.std(ibs_cv):.4f} '
          f'{resultados_test[m]["C-index"]:>14.4f} '
          f'{resultados_test[m]["IBS"]:>10.4f}')
print('  ' + '─' * 60)
print(f'  {"KM (ref)":<12} {"—":>16} {"—":>14} {"—":>14} {IBS_KM_REF:>10.4f}')
print('═' * 65)
print(f'\n  🏆 Mayor C-index test : {mejor_ci[0]}  ({mejor_ci[1]["C-index"]:.4f})')
print(f'  🏆 Menor IBS test     : {mejor_ibs[0]}  ({mejor_ibs[1]["IBS"]:.4f})')
print('═' * 65)
print()
print('  Umbrales de referencia (literatura oncológica):')
print('  · C-index > 0.70  → discriminación aceptable')
print('  · C-index > 0.75  → discriminación buena')
print('  · IBS < 0.15      → calibración buena')
print('  · IBS < 0.10      → calibración excelente')